# HealthIQ — 01: Data Cleaning

**Objective:** Inspect the raw dataset, document all quality issues, apply cleaning transformations, and save the cleaned dataset.

**Input:** `../data/raw/healthcare_patient_analytics_seaborn.csv`  
**Output:** `../data/processed/cleaned_healthcare_data.csv`

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Raw Data

In [5]:
df_raw = pd.read_csv('../data/raw/healthcare_patient_analytics_seaborn.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(10)

Shape: (5000, 12)


,patient_id,visit_date,age_group,gender,region,department,treatment_type,visit_type,length_of_stay_days,treatment_cost,recovery_score,readmission_risk
0,1,2022-01-01 00:00:00,31-45,Male,West,General Medicine,Medication,Emergency,5.80,59151.00,59.00,0.12
1,2,2022-01-01 01:00:00,60+,Female,West,Orthopedics,Surgery,Routine,5.10,30272.00,97.00,0.11
2,3,2022-01-01 02:00:00,46-60,Male,South,Pediatrics,Observation,Routine,7.90,67498.00,60.00,0.19
3,4,2022-01-01 03:00:00,31-45,Female,North,Neurology,Medication,Routine,5.00,29896.00,51.00,0.47
4,5,2022-01-01 04:00:00,18-30,Female,North,Neurology,Therapy,Routine,0.00,36208.00,60.00,0.40
5,6,2022-01-01 05:00:00,18-30,Male,North,Neurology,Medication,Routine,5.70,34016.00,96.00,0.07
6,7,2022-01-01 06:00:00,18-30,Female,North,Cardiology,Surgery,Routine,0.80,51496.00,53.00,0.10
7,8,2022-01-01 07:00:00,60+,Female,South,Cardiology,Therapy,Emergency,4.90,47951.00,64.00,0.73
8,9,2022-01-01 08:00:00,46-60,Male,South,Neurology,Observation,Emergency,0.60,54975.00,58.00,0.04
9,10,2022-01-01 09:00:00,46-60,Male,South,General Medicine,Medication,Routine,3.10,36895.00,72.00,0.32


## 2. Basic Inspection

In [6]:
# Data types
df_raw.dtypes

patient_id               int64
visit_date              object
age_group               object
gender                  object
region                  object
department              object
treatment_type          object
visit_type              object
length_of_stay_days    float64
treatment_cost         float64
recovery_score         float64
readmission_risk       float64
dtype: object

In [7]:
# Statistical summary
df_raw.describe(include='all')

,patient_id,visit_date,age_group,gender,region,department,treatment_type,visit_type,length_of_stay_days,treatment_cost,recovery_score,readmission_risk
count,5000.00,5000,5000,5000,5000,5000,5000,5000,5000.00,5000.00,5000.00,5000.00
unique,NaN,5000,4,2,4,5,4,2,NaN,NaN,NaN,NaN
top,NaN,2022-07-28 07:00:00,31-45,Male,West,Orthopedics,Observation,Routine,NaN,NaN,NaN,NaN
freq,NaN,1,1716,2508,1290,1058,1270,3432,NaN,NaN,NaN,NaN
mean,2500.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.06,54915.47,74.72,0.28
std,1443.52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.93,19481.16,11.87,0.16
min,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,746.00,33.00,0.01
25%,1250.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.70,41244.75,67.00,0.16
50%,2500.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.00,55123.50,75.00,0.26
75%,3750.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.40,68012.00,83.00,0.38


## 3. Missing Values

In [8]:
missing = df_raw.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing: {missing.sum()}')

Missing values per column:
patient_id             0
visit_date             0
age_group              0
gender                 0
region                 0
department             0
treatment_type         0
visit_type             0
length_of_stay_days    0
treatment_cost         0
recovery_score         0
readmission_risk       0
dtype: int64

Total missing: 0


**Finding:** No missing values. No imputation required.

## 4. Duplicate Rows

In [9]:
dups = df_raw.duplicated().sum()
print(f'Duplicate rows: {dups}')

Duplicate rows: 0


**Finding:** No duplicates found.

## 5. Categorical Column Inspection

In [10]:
cat_cols = ['age_group', 'gender', 'region', 'department', 'treatment_type', 'visit_type']
for col in cat_cols:
    print(f'--- {col} ---')
    print(df_raw[col].value_counts())
    has_space = (df_raw[col].str.strip() != df_raw[col]).sum()
    print(f'Whitespace issues: {has_space}\n')

--- age_group ---
age_group
31-45    1716
18-30    1283
46-60    1271
60+       730
Name: count, dtype: int64
Whitespace issues: 0

--- gender ---
gender
Male      2508
Female    2492
Name: count, dtype: int64
Whitespace issues: 0

--- region ---
region
West     1290
North    1282
East     1217
South    1211
Name: count, dtype: int64
Whitespace issues: 0

--- department ---
department
Orthopedics         1058
Cardiology           995
General Medicine     991
Pediatrics           989
Neurology            967
Name: count, dtype: int64
Whitespace issues: 0

--- treatment_type ---
treatment_type
Observation    1270
Surgery        1262
Medication     1236
Therapy        1232
Name: count, dtype: int64
Whitespace issues: 0

--- visit_type ---
visit_type
Routine      3432
Emergency    1568
Name: count, dtype: int64
Whitespace issues: 0



## 6. Numerical Validation

In [11]:
print('Negative length_of_stay_days:', (df_raw['length_of_stay_days'] < 0).sum())
print('Negative treatment_cost:', (df_raw['treatment_cost'] < 0).sum())
print('Invalid recovery_score (<0 or >100):', ((df_raw['recovery_score'] < 0) | (df_raw['recovery_score'] > 100)).sum())
print()
print('readmission_risk range:', df_raw['readmission_risk'].min(), 'to', df_raw['readmission_risk'].max())

Negative length_of_stay_days: 0
Negative treatment_cost: 0
Invalid recovery_score (<0 or >100): 0

readmission_risk range: 0.01 to 0.84


## 7. Outlier Detection

In [12]:
num_cols = ['length_of_stay_days', 'treatment_cost', 'recovery_score']
for col in num_cols:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR
    outliers = df_raw[(df_raw[col] < low) | (df_raw[col] > high)]
    print(f'{col}: {len(outliers)} outliers | bounds [{low:.2f}, {high:.2f}]')

length_of_stay_days: 15 outliers | bounds [-1.35, 9.45]
treatment_cost: 15 outliers | bounds [1093.88, 108162.88]
recovery_score: 18 outliers | bounds [43.00, 107.00]


**Decision:** All outliers are clinically plausible (e.g., LOS up to 11.9 days, costs up to \$119k). They are **retained**.

## 8. Apply Cleaning Transformations

In [13]:
df = df_raw.copy()

# 1. Parse visit_date to datetime
df['visit_date'] = pd.to_datetime(df['visit_date'])

# 2. Strip whitespace from all categorical columns (defensive)
for col in cat_cols:
    df[col] = df[col].str.strip()

# 3. Bin readmission_risk (0–1 float) into Low / Medium / High
bins = [0, 0.3, 0.6, 1.0]
labels = ['Low', 'Medium', 'High']
df['readmission_risk'] = pd.cut(
    df['readmission_risk'], bins=bins, labels=labels, include_lowest=True
).astype(str)

print('Cleaning complete.')
print('Shape:', df.shape)
print('visit_date dtype:', df['visit_date'].dtype)
print('readmission_risk values:', df['readmission_risk'].value_counts().to_dict())

Cleaning complete.
Shape: (5000, 12)
visit_date dtype: datetime64[ns]
readmission_risk values: {'Low': 3002, 'Medium': 1822, 'High': 176}


## 9. Final Dataset Summary

In [14]:
print(f'Original rows: {len(df_raw)}')
print(f'Cleaned rows:  {len(df)}')
print(f'Rows removed:  {len(df_raw) - len(df)}')
print(f'Date range:    {df["visit_date"].min().date()} to {df["visit_date"].max().date()}')
df.head()

Original rows: 5000
Cleaned rows:  5000
Rows removed:  0
Date range:    2022-01-01 to 2022-07-28


,patient_id,visit_date,age_group,gender,region,department,treatment_type,visit_type,length_of_stay_days,treatment_cost,recovery_score,readmission_risk
0,1,2022-01-01 00:00:00,31-45,Male,West,General Medicine,Medication,Emergency,5.80,59151.00,59.00,Low
1,2,2022-01-01 01:00:00,60+,Female,West,Orthopedics,Surgery,Routine,5.10,30272.00,97.00,Low
2,3,2022-01-01 02:00:00,46-60,Male,South,Pediatrics,Observation,Routine,7.90,67498.00,60.00,Low
3,4,2022-01-01 03:00:00,31-45,Female,North,Neurology,Medication,Routine,5.00,29896.00,51.00,Medium
4,5,2022-01-01 04:00:00,18-30,Female,North,Neurology,Therapy,Routine,0.00,36208.00,60.00,Medium


## 10. Save Cleaned Dataset

In [15]:
df.to_csv('../data/processed/cleaned_healthcare_data.csv', index=False)
print('Saved: ../data/processed/cleaned_healthcare_data.csv')

Saved: ../data/processed/cleaned_healthcare_data.csv
